# OS-Tutor: QLoRA Fine-Tuning Pipeline for Qwen2.5-3B-Instruct

**Project:** OS-Tutor — AI Teaching Assistant for Operating Systems  
**Team:** Two-Person College Project  
**Objective:** Fine-tune Qwen2.5-3B-Instruct with 4-bit QLoRA to mimic a professor's teaching style (Definition → Explanation → Example → Exam Tip).

---
### Step 1: Install Pinned Dependencies
Fixes known `trl` and `transformers` argument mismatch by pinning exact compatible versions.

In [ ]:
# Install pinned libraries to avoid SFTConfig / SFTTrainer mismatch
!pip install -q \
    trl==0.11.4 \
    transformers==4.46.0 \
    peft==0.13.2 \
    bitsandbytes==0.44.1 \
    accelerate==1.0.1 \
    datasets==3.0.1

### Step 2: Verify GPU and Imports

In [ ]:
import torch
import transformers
import trl
import peft
import bitsandbytes

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
print(f"Transformers: {transformers.__version__}")
print(f"TRL: {trl.__version__}")
print(f"PEFT: {peft.__version__}")

### Step 3: Load Dataset (train.json & val.json)

In [ ]:
import json
from datasets import Dataset

# Format function for Alpaca template
def format_prompts(batch):
    texts = []
    for instruction, input_text, output in zip(batch["instruction"], batch["input"], batch["output"]):
        if input_text and input_text.strip():
            text = f"### Instruction:\n{instruction.strip()}\n\n### Input:\n{input_text.strip()}\n\n### Response:\n{output.strip()}"
        else:
            text = f"### Instruction:\n{instruction.strip()}\n\n### Response:\n{output.strip()}"
        texts.append(text)
    return {"text": texts}

# Load dataset from local file or upload
with open("train.json", "r") as f:
    train_data = json.load(f)
with open("val.json", "r") as f:
    val_data = json.load(f)

train_ds = Dataset.from_list(train_data).map(format_prompts, batched=True)
val_ds = Dataset.from_list(val_data).map(format_prompts, batched=True)

print(f"Train samples: {len(train_ds)}, Val samples: {len(val_ds)}")
print("Sample prompt:")
print(train_ds[0]["text"]) 

### Step 4: Configure 4-bit BitsAndBytes Quantization & Load Base Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

model_id = "Qwen/Qwen2.5-3B-Instruct"

# NF4 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)
print("Base model loaded and prepared for 4-bit QLoRA training!")

### Step 5: Configure LoRA Adapter (Targeting q, k, v, o projections)

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

### Step 6: Train with SFTConfig & SFTTrainer
Here we pass `dataset_text_field` and `max_seq_length` to `SFTConfig`, resolving the argument warning/error in TRL.

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    evaluation_strategy="epoch",
    fp16=True,
    dataset_text_field="text",
    max_seq_length=512,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=peft_config,
    tokenizer=tokenizer,
    args=training_args,
)

train_result = trainer.train()
print(train_result)

### Step 7: Export LoRA Adapter and Download
Saves the adapter to `os-tutor-adapter/` and compresses it into a zip file for use in the local project.

In [ ]:
adapter_dir = "os-tutor-adapter"
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

!zip -r os-tutor-adapter.zip os-tutor-adapter/

# Download to local computer
from google.colab import files
files.download("os-tutor-adapter.zip")